In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import re

In [2]:
url = 'https://en.wikipedia.org/wiki/List_of_songs_recorded_by_Nirvana'
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

In [3]:
table = soup.find_all('table', class_ = 'wikitable sortable plainrowheaders')[0]

column_names = ['song', 'writers', 'original_release', 'producers', 'year']

songs = []
writers = []
original = []
producers = []
year = []

rows = table.find_all('tr')

for row in rows[1:]:    
    if row.th.string:
        songs.append(row.th.string.replace("\n","").replace('"','').strip())
        continue
        
    if row.th.a:
        songs.append(row.th.a.string.replace("\n","").replace('"','').strip())
        continue
        
    if row.th.contents[0].name == 'span':
        if row.th.span.string == None:
            songs.append(row.th.contents[1].replace("\n","").replace('"','').strip())
            continue
        else:
            songs.append(row.th.contents[0].string.replace("\n","").replace('"','').strip())
            continue
    songs.append(row.th.contents[0].string.replace("\n","").replace('"','').strip())

for row in rows:
    if row.td and row.td.a:
        writers.append(', '.join([string for string in row.td.strings if string.strip() 
                                  and not 'note' in string 
                                  and not ' (' in string 
                                  and not 'Unleashed' in string 
                                  and not 'cover' in string]))
    if row.td and not row.td.a:
        if not row.td.span:
            writers.append(row.td.string.strip())
        else:
            writers.append(row.td.span.string.strip())

for row in rows[1:]:
    data = row.find_all('td')[1]
    if 'Non-album single' in ''.join([string for string in data.strings]):
        original.append('Non-album single')
        continue
        
    if data.i and data.i.a:
        original.append(data.i.a.string)
        continue
    elif data.i:
        original.append(data.i.string)
        continue
    else:
        original.append('Unknown')
        
for row in rows[1:]:
    
    data = row.find_all('td')[2]
    
    if '–' in str(data):
        producers.append('Unknown')
        continue
        
    if data.br:
        p = []
        for a in data.find_all('a'):
            p.append(a.string)
        producers.append(', '.join(p))
        continue
        
    if data.span and data.span.a:
        producers.append(data.span.a.string)
        continue
    
    if data.span:
        producers.append(data.span.string)
        continue
        
    if not data.string:
        producers.append('Unknown')
        continue
  
for row in rows[1:]:
    data = row.find_all('td')[3]
    year.append((int(data.string[0:4]) if data.string else -1))

    
print(len(writers))
print(len(songs))
print(len(original))
print(len(producers))
print(len(year))

124
124
124
124
124


In [4]:
df = pd.DataFrame({
    'songs': songs,
    'writers' : writers,
    'original_release' : original,
    'producers' : producers,
    'release_year': year
})

df

,songs,writers,original_release,producers,release_year
0,About a Girl,Kurt Cobain,Bleach,Jack Endino,1989
1,Aero Zeppelin,Kurt Cobain,Incesticide,Jack Endino,1992
2,Ain't It a Shame,Traditional,With the Lights Out,Jack Endino,2004
3,All Apologies,Kurt Cobain,In Utero,Steve Albini,1993
4,Aneurysm,"Kurt Cobain, Dave Grohl, Krist Novoselic",Non-album single,Craig Montgomery,1991
...,...,...,...,...,...
119,You Know You're Right,Kurt Cobain,Nirvana,Adam Kasper,2002
120,Untitled,Unknown,Unknown,Unknown,-1
121,Untitled,Unknown,Unknown,Unknown,-1
122,Untitled,Unknown,Unknown,Unknown,-1


In [5]:
df.to_csv('nirvana_wiki.csv')